# Qwen 3.5 9B Tool-Calling Fine-Tune Cloud Stable
This cloud-stable variant keeps the same 31B / 4-bit / 4K-context quality target, but uses safer Unsloth-aligned settings to reduce notebook kernel crash risk.
**Changes**
- LoRA rank reduced from 32 to 16
- Unsloth gradient checkpointing enabled
- bounded `max_steps` chunk instead of a long 2-epoch run
- checkpoints every 10 steps
- first pass uses a smaller dataset slice


### Installation


In [ ]:
# === Create a clean AMD/Unsloth kernel (doc-aligned path) ===
# This follows the official AMD + Unsloth guidance: isolated env, ROCm PyTorch, then Unsloth.
import os, re, shutil, subprocess, sys
from pathlib import Path
INSTALL_ROOT = os.environ.get('UNSLOTH_INSTALL_ROOT', str(Path.home() / 'unsloth-qwen35'))
VENV = os.path.join(INSTALL_ROOT, 'venv')
TMPDIR = os.path.join(INSTALL_ROOT, 'tmp')
PIP_CACHE_DIR = os.path.join(INSTALL_ROOT, 'pip-cache')
PY = os.path.join(VENV, 'bin', 'python')
PIP = [PY, '-m', 'pip']
def run(cmd):
    print('\n$ ' + ' '.join(cmd))
    env = os.environ.copy()
    env['TMPDIR'] = TMPDIR
    env['TEMP'] = TMPDIR
    env['TMP'] = TMPDIR
    env['PIP_CACHE_DIR'] = PIP_CACHE_DIR
    r = subprocess.run(cmd, capture_output=True, text=True, env=env)
    if r.stdout.strip():
        out = r.stdout.strip().splitlines()
        for line in out[-8:]:
            print(line)
    if r.returncode != 0:
        err = r.stderr.strip() or '(no stderr)'
        raise RuntimeError(err)
def detect_rocm_tag():
    candidates = []
    try:
        r = subprocess.run(['amd-smi', 'version'], capture_output=True, text=True)
        candidates.append(r.stdout)
    except FileNotFoundError:
        pass
    try:
        candidates.append(Path('/opt/rocm/.info/version').read_text())
    except Exception:
        pass
    try:
        r = subprocess.run(['hipconfig', '--version'], capture_output=True, text=True)
        candidates.append(r.stdout)
    except FileNotFoundError:
        pass
    for text in candidates:
        m = re.search(r'(?:ROCm version: |HIP version: )([0-9]+)\.([0-9]+)', text)
        if not m:
            m = re.search(r'([0-9]+)\.([0-9]+)', text)
        if m:
            return f'rocm{m.group(1)}.{m.group(2)}'
    return 'rocm7.0'
Path(INSTALL_ROOT).mkdir(parents=True, exist_ok=True)
Path(TMPDIR).mkdir(parents=True, exist_ok=True)
Path(PIP_CACHE_DIR).mkdir(parents=True, exist_ok=True)
rocm_tag = detect_rocm_tag()
print('ROCm wheel tag:', rocm_tag)
print('Install root:', INSTALL_ROOT)
print('Temp dir:', TMPDIR)
print('Pip cache:', PIP_CACHE_DIR)
if os.path.isdir(VENV):
    shutil.rmtree(VENV)
run([sys.executable, '-m', 'venv', VENV])
run(PIP + ['install', '--upgrade', 'pip', 'setuptools', 'wheel', 'ipykernel'])
run(PIP + ['install', '--upgrade', '--force-reinstall', 'torch', 'torchvision', 'torchaudio', '--index-url', f'https://download.pytorch.org/whl/{rocm_tag}'])
run(PIP + ['install', '--no-deps', 'unsloth', 'unsloth-zoo'])
run(PIP + ['install', '--no-deps', 'git+https://github.com/unslothai/unsloth-zoo.git'])
run(PIP + ['install', 'unsloth[amd] @ git+https://github.com/unslothai/unsloth'])
run(PIP + ['install', '--upgrade', 'git+https://github.com/huggingface/transformers.git'])
run(PIP + ['install', '--upgrade', 'timm'])
run([PY, '-m', 'ipykernel', 'install', '--user', '--name', 'unsloth-qwen-amd', '--display-name', 'Python (unsloth-qwen-amd)'])
print('\nKernel created: Python (unsloth-qwen-amd)')
print('Next: Kernel -> Change Kernel -> Python (unsloth-qwen-amd)')
print('Then restart kernel and run Cell 3.')


In [ ]:
# === Verify you are on the clean Unsloth AMD kernel ===
import os, sys
from pathlib import Path
VENV = os.environ.get('UNSLOTH_VENV', str(Path.home() / 'unsloth-qwen35' / 'venv'))
print('sys.executable:', sys.executable)
print('sys.prefix:', sys.prefix)
print('VIRTUAL_ENV:', os.environ.get('VIRTUAL_ENV'))
on_expected_kernel = (
    sys.prefix == VENV
    or sys.executable.startswith(VENV + os.sep)
    or os.environ.get('VIRTUAL_ENV') == VENV
)
if not on_expected_kernel:
    raise RuntimeError(
        'Wrong kernel. Change kernel to Python (unsloth-qwen-amd), then restart and re-run this cell.'
    )

import unsloth, unsloth_zoo, huggingface_hub, regex, transformers
from transformers import AutoConfig

print(f'unsloth: {getattr(unsloth, "__version__", "OK")} ({unsloth.__file__})')
print(f'unsloth_zoo: {getattr(unsloth_zoo, "__version__", "OK")} ({unsloth_zoo.__file__})')
print(f'huggingface_hub: {huggingface_hub.__version__} ({huggingface_hub.__file__})')
print(f'regex: {regex.__version__} ({regex.__file__})')
print(f'transformers: {transformers.__version__} ({transformers.__file__})')
c = AutoConfig.from_pretrained("unsloth/Qwen3.5-9B")
print(f'AutoConfig OK: model_type={c.model_type}')

import torch; torch._dynamo.config.recompile_limit = 64
print('Next: run the HF token cell, then the Load Model cell.')

In [ ]:
import os
from pathlib import Path
import huggingface_hub
# ==========================================================
# ⚠️ PASTE YOUR HUGGING FACE TOKEN BETWEEN THE QUOTES BELOW:
# ==========================================================
HF_TOKEN = ""
HF_REPO_ID = "mdtita/Qwen3.5-9B-Arabic-Agent-SFT"
# Fallback to environment variable if the string above is empty
if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN or HF_TOKEN == "YOUR_HF_TOKEN":
    print("⚠️  No HF token set. Hub pushing disabled, and downloading gated models may fail!")
    print("   Please paste your token in the HF_TOKEN variable above.")
    HF_TOKEN = ""
    PUSH_TO_HUB = False
else:
    print("✅ Hugging Face Token recognized!")
    huggingface_hub.login(token=HF_TOKEN)
    PUSH_TO_HUB = True
# === Training Output ===
TRAINING_OUTPUT_DIR = os.environ.get(
    "TRAINING_OUTPUT_DIR",
    str(Path.home() / "outputs_qwen35_9B_arabic_sft")
)
Path(TRAINING_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
# Point compiler caches to the syncable folder
os.environ['TRITON_CACHE_DIR'] = os.path.join(TRAINING_OUTPUT_DIR, 'triton_cache')
os.environ['TORCHINDUCTOR_CACHE_DIR'] = os.path.join(TRAINING_OUTPUT_DIR, 'torch_cache')
print("HF repo:", HF_REPO_ID)
print("Push enabled:", PUSH_TO_HUB)
print("Training outputs:", TRAINING_OUTPUT_DIR)


### Load Model


In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048  # Must be defined before dataset formatting

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3.5-9B",
    dtype = torch.bfloat16,
    max_seq_length = max_seq_length,
    load_in_4bit = False,
    full_finetuning = False,
    token = HF_TOKEN or None,
)

### Add LoRA Adapters


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",
                      "out_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    max_seq_length = max_seq_length,
)


### Data Prep

We use the `qwen-2.5` chat template. Qwen 3.5 renders conversations using ChatML:
```
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Hello<|im_end|>
<|im_start|>assistant
Hey there!<|im_end|>
```

In [ ]:
# Qwen 3.5 tokenizer already includes the correct chat template.
# No get_chat_template() call needed — just use tokenizer.apply_chat_template() directly.
# Ref: https://github.com/unslothai/notebooks/blob/main/nb/Qwen_3_5_27B_A100(80GB).ipynb
print("Chat template:", tokenizer.chat_template[:80], "...")


In [ ]:
from datasets import load_dataset

dataset = load_dataset("mtita/qwen3.5-arabic-agent-dataset", split="train")
print(f"Loaded {len(dataset)} training examples")
print(dataset[0]["messages"][:2])

Apply the Qwen chat template and tokenize.

In [ ]:
def formatting_prompts_func(examples):
    texts = []
    for msgs in examples["messages"]:
        texts.append(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False))
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True, remove_columns=dataset.column_names, num_proc=16)
print(f"Formatted: {len(dataset)} examples")


### Stability strategy
Training runs for **1 full epoch**. Checkpoints are saved every 400 steps. If the kernel dies (50-min timeout), re-run the setup cells then the training cell — it will auto-resume from the last checkpoint via the chunking loop.

### Train the model


In [ ]:
from transformers.trainer_utils import get_last_checkpoint
LAST_CHECKPOINT = get_last_checkpoint(TRAINING_OUTPUT_DIR)
print("Latest checkpoint:", LAST_CHECKPOINT or "none")


Only train on assistant outputs, ignore user inputs:


In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        dataset_num_proc = 16,
        packing = True,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 20,
        learning_rate = 2e-5,
        num_train_epochs = 1,
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = TRAINING_OUTPUT_DIR,
        save_strategy = "steps",
        save_steps = 200,
        save_total_limit = 3,
        report_to = "none",
        bf16 = True,
    ),
)


In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)


### Verify masking before training
Run these to confirm only assistant responses are being trained on.


In [ ]:
# Verify masking - should see full conversation
tokenizer.decode(trainer.train_dataset[0]["input_ids"])


In [ ]:
# Verify masking - should see ONLY assistant responses (rest is spaces)
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[0]["labels"]]).replace(tokenizer.pad_token, " ")


### Start or resume training
If the kernel dies or the notebook disconnects, re-open the notebook, run the setup cells again, then run the cell below.


In [ ]:
import gc
import os
import json
import torch
from huggingface_hub import snapshot_download, HfApi
from transformers.trainer_utils import get_last_checkpoint
def get_step_from_checkpoint(ckpt_dir):
    if not ckpt_dir: return 0
    try:
        with open(os.path.join(ckpt_dir, "trainer_state.json")) as f:
            return json.load(f).get("global_step", 0)
    except:
        try: return int(ckpt_dir.split("-")[-1])
        except: return 0
# 0. Sync missing checkpoints from the Hub
if PUSH_TO_HUB:
    print(f"\n📥 Pulling rescued checkpoints from Hugging Face: {HF_REPO_ID}...")
    try:
        snapshot_download(
            repo_id=HF_REPO_ID,
            local_dir=TRAINING_OUTPUT_DIR,
            token=HF_TOKEN,
            allow_patterns=["*checkpoint*", "triton_cache/*", "torch_cache/*"] # ALWAYS pull everything available
        )
        print("✅ Cloud checkpoints restored to the fresh VM!")
    except Exception as e:
        print("No prior checkpoints found on Hub, starting fresh.")
trainer.args.save_strategy = "steps"
trainer.args.save_steps = 400              # DROPPED TO 600 FOR HUGE SAFETY MARGIN
trainer.args.save_total_limit = 2
trainer.args.push_to_hub = False           # OFF: KILLS THE FLAKY ASYNC BACKGROUND THREAD
CHUNK_SIZE = 50                           # DROPPED TO 600
TOTAL_CHUNKS = 150                          # UPPED TO COVER FULL 15K
for i in range(TOTAL_CHUNKS):
    LAST_CHECKPOINT = get_last_checkpoint(TRAINING_OUTPUT_DIR)
    last_hf_ckpt = os.path.join(TRAINING_OUTPUT_DIR, "last-checkpoint")
    if not LAST_CHECKPOINT and os.path.exists(last_hf_ckpt):
        LAST_CHECKPOINT = last_hf_ckpt
    current_max_steps = (i + 1) * CHUNK_SIZE
    trainer.args.max_steps = current_max_steps
    completed_steps = get_step_from_checkpoint(LAST_CHECKPOINT)
    if LAST_CHECKPOINT and completed_steps >= current_max_steps:
        print(f"⏭️ Skipping chunk {i+1} (Already completed in a previous run)")
        continue
    print(f"\n🚀 Running Chunk {i+1}: Training up to step {current_max_steps}...")
    try:
        trainer_stats = trainer.train(resume_from_checkpoint=LAST_CHECKPOINT)
    except Exception as e:
        print(f"Training interrupted or hit an error: {e}")
        break
    # 100% Dedicated Synchronous Upload
    if PUSH_TO_HUB:
        print(f"📦 Chunk {i+1} finished! Force-uploading local checkpoints to the Cloud...")
        api = HfApi(token=HF_TOKEN)
        api.upload_folder(
            folder_path=TRAINING_OUTPUT_DIR,
            repo_id=HF_REPO_ID,
            repo_type="model",
            allow_patterns=["checkpoint-*/*", "triton_cache/*", "torch_cache/*"], # ONLY upload the pure numbered checkpoint folders!
            commit_message=f"Automated sync reaching step {current_max_steps}"
        )
        print("☁️ Cloud upload totally complete and secured!")
    gc.collect()
    torch.cuda.empty_cache()
    print(f"🧹 Chunk {i+1} Memory flushed. Moving to next chunk.")


### Push final adapter to Hugging Face
Use this after training completes so you can always restore from Hub even if the cloud VM is wiped.


In [ ]:
if PUSH_TO_HUB:
    trainer.push_to_hub()
    tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
    print("Pushed final training artifacts to", HF_REPO_ID)
else:
    print("Skipping push_to_hub")


In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")


### Test Inference


In [ ]:
# Quick inference test
messages = [{"role": "user",
"content": "You have these tools:\n"
"- read(path): Read a file\n"
"- edit(path, old_text, new_text): Replace text in a file\n"
"- bash(command): Run a shell command\n\n"
"Fix the typo in src/utils.py where 'retrun' should be 'return'."}]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 512,
                    temperature = 0.7, top_p = 0.9)

### Save LoRA adapters
**Set your HF token below!**


In [ ]:
model.save_pretrained("qwen35_9b_arabic_lora")
tokenizer.save_pretrained("qwen35_9b_arabic_lora")
if PUSH_TO_HUB:
    model.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
    print("Pushed LoRA adapter to", HF_REPO_ID)
else:
    print("Saved locally. Set HF_TOKEN to push to Hub.")

### Export to GGUF
Supported: q4_k_m (recommended), q5_k_m, q8_0, f16, bf16 and more.


In [ ]:
model.save_pretrained_gguf(
    "qwen35_9b_arabic_agent_gguf",
    tokenizer,
    quantization_method = "q4_k_m",
)

In [ ]:
# Push GGUF to HuggingFace
if PUSH_TO_HUB:
    model.push_to_hub_gguf(
        HF_REPO_ID + "-GGUF",
        tokenizer,
        quantization_method = "q4_k_m",
        token = HF_TOKEN,
    )
    print("Pushed GGUF to Hub")
else:
    print("Saved GGUF locally. Set HF_TOKEN to push.")